<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_Phase2_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Gemma-Titans Phase 2: LM Fine-Tuning

Фаза 2 обучения: переключаемся с послойной дистилляции на language modeling loss.

**Отличия от Phase 1:**
- Выходы студента передаются между слоями (нет `stop_gradient`, нет teacher chain)
- Loss: cross-entropy по токенам вместо per-layer MSE
- `training_phase=2` в конфиге модели
- Веса загружаются из `saved_titans_delta` (результат Phase 1)
- Сниженные LR (веса уже предобучены)

In [2]:
# 0. Environment Setup
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog
!pip install -q flax optax seqio
!pip install importlib_resources

Cloning into 'kauldron'...
remote: Enumerating objects: 471, done.
remote: Counting objects: 100% (471/471), done.
remote: Compressing objects: 100% (404/404), done.
remote: Total 471 (delta 82), reused 327 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (471/471), 706.39 KiB | 2.43 MiB/s, done.
Resolving deltas: 100% (82/82), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 57.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.7 MB/s eta 0:00:00
   

In [3]:
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git

Cloning into 'Titans_jax'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 0), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 83.68 KiB | 1.12 MiB/s, done.


## Start

In [4]:
import os
# os._exit(0)

In [1]:
import sys
import os
sys.path.append(os.getcwd())

import jax
import jax.numpy as jnp
import optax
import dataclasses
import numpy as np
import os
import orbax.checkpoint as ocp
import shutil

from kauldron import kd
from kauldron.ktyping import Array
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd /content/Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

/content/Titans_jax
JAX Backend: cpu
Devices: [CpuDevice(id=0)]
JAX Backend: cpu
Devices: [CpuDevice(id=0)]


In [2]:
import psutil
import os

# Get system memory stats
ram_info = psutil.virtual_memory()
total_ram_gb = ram_info.total / (1024**3)
available_ram_gb = ram_info.available / (1024**3)

print(f"Total RAM: {total_ram_gb:.2f} GB")
print(f"Available RAM: {available_ram_gb:.2f} GB")

# Check if High-RAM is likely active
if total_ram_gb > 170:
    print("\nStatus: Likely High-RAM runtime")
else:
    print("\nStatus: Standard RAM runtime")



Total RAM: 12.67 GB
Available RAM: 10.93 GB

Status: Standard RAM runtime


## 1. Загрузка весов из Phase 1

In [7]:
!cp "/content/drive/Shareddrives/shared_veriga/saved_titans_delta/saved_titans_phase2_from_11.zip" saved_titans_delta.zip

## 2. Гиперпараметры

In [3]:
batch_size = 4
max_length = 2048
total_steps = 60000

titans_phase2_first_layer = 11  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

# Гиперпараметры Titans Memory
huber_loss_delta = optax.constant_schedule(0.5)
# optax.linear_schedule(
#    init_value=0.1, end_value=.5, transition_begin = 100, transition_steps=19900)
# )
# huber_loss_delta = optax.cosine_decay_schedule(
#     init_value=0.8,
#     decay_steps=10000,
#     alpha=0.1,
# )
  # Можно передать число или optax schedule

_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")


Active Titans layers: (11, 17, 23)


In [9]:
huber_loss_delta(90000)

0.5

In [10]:
def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

titans_zip = "saved_titans_delta.zip"
titans_delta_path = "./saved_titans_delta"

if os.path.exists(titans_zip) and not os.path.exists(titans_delta_path):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

print("Loading Gemma base weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

print("Loading Phase 1 Titans weights...")
loaded_titans_params = load_titans_weights(titans_delta_path)

# Keep only weights for active Titans layers — inactive layers (< titans_phase2_first_layer)
# revert to original Gemma weights automatically via merge.
active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}
print(f"Merging Titans weights for: {sorted(active_layer_keys)}")

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
print("✅ Phase 1 weights loaded and merged.")

Unpacking saved_titans_delta.zip...
Loading Gemma base weights...


Loading Phase 1 Titans weights...
Merging Titans weights for: ['layer_11', 'layer_17', 'layer_23']
✅ Phase 1 weights loaded and merged.


In [ ]:
# !pip install apache-beam[interactive]

## 3. Модель (training_phase=2)

In [4]:

gemma_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    # sliding_window_size=128,
    training_phase=2,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
    neural_mem_huber_delta=huber_loss_delta,
)

model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
    # step="step",
)

## 4. Датасет

In [47]:
import numpy as np
import glob
import os
from kauldron import kd
import sys

tokenizer = gm.text.Gemma3Tokenizer()
# Using the path discovered earlier to avoid TFDS/Beam logic
EXTRACTED_TEXT_DIR = "/content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted/ZIP.OpenWebText.zip/OpenWebText/Version 1/URLs"

class GeneratorWithLen:
    """Wrapper to provide a length and subscriptability for Kauldron/Grain compatibility."""
    def __init__(self, gen_fn, length=10**12):
        self.gen_fn = gen_fn
        self.length = length
        self._iterator = iter(self.gen_fn())

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        # PyGrain MapDataset requires random access. We bypass it for streaming
        # by ignoring the index and yielding the next element.
        try:
            return next(self._iterator)
        except StopIteration:
            self._iterator = iter(self.gen_fn())
            return next(self._iterator)

def get_clm_dataset(batch_size, tokenizer, max_length, split="train"):
    """Manual generator dataset that avoids indexing issues by using a pure Python iterable."""

    text_files = sorted(glob.glob(os.path.join(EXTRACTED_TEXT_DIR, "*.txt")))
    if not text_files:
        raise FileNotFoundError(f"No text files found in {EXTRACTED_TEXT_DIR}")

    def pure_python_generator():
        while True:
            np.random.shuffle(text_files)
            for f_path in text_files:
                try:
                    with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                        # Split into rough document chunks
                        for doc in content.split('\n\n'):
                            doc = doc.strip()
                            if len(doc) < 10: continue

                            # Tokenize
                            tokens = tokenizer.encode(doc, add_bos=True)[:max_length]
                            pad_len = max_length - len(tokens)
                            tokens = np.pad(np.array(tokens, dtype=np.int32), (0, pad_len))

                            yield {'tokens': tokens}
                except Exception:
                    continue

    # Wrap the generator function to provide a length and __getitem__
    wrapped_iterable = GeneratorWithLen(pure_python_generator)

    return kd.data.py.DataSource(
        wrapped_iterable,
        batch_size=batch_size,
        shuffle=False,
        num_epochs=1,
        shard_by_process=False,
    )

### Создание кэша (выполнить один раз)
Этот код прочитает текстовые файлы, токенизирует их и сохранит в `.npy` файл на Google Диск. В следующий раз вы сможете просто загружать этот файл.

In [59]:
import numpy as np
from tqdm.auto import tqdm
import glob
import os
import gc

CACHE_DIR = "/content/drive/Shareddrives/shared_veriga/tfds_cache/pretokenized_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_FILE = os.path.join(CACHE_DIR, "openwebtext_tokens_2048.npy")

def build_tokenized_cache(target_docs=1000000):
    text_files = sorted(glob.glob(os.path.join(EXTRACTED_TEXT_DIR, "*.txt")))

    # Проверяем, есть ли уже сохраненный кэш
    if os.path.exists(CACHE_FILE):
        print(f"Loading existing cache from {CACHE_FILE}...")
        existing_tokens = np.load(CACHE_FILE)
        all_tokens = existing_tokens.tolist()
        docs_processed = len(all_tokens)
        print(f"Resuming from {docs_processed} documents.")
        del existing_tokens
        gc.collect() # Очищаем память
    else:
        all_tokens = []
        docs_processed = 0

    if docs_processed >= target_docs:
        print("Target number of documents already reached!")
        return

    print(f"Reading and tokenizing texts to save in {CACHE_FILE}...")

    # Счетчик для пропуска уже обработанных документов в файлах
    docs_skipped = 0

    with tqdm(total=target_docs, initial=docs_processed, desc="Processing documents") as pbar:
        for f_path in text_files:
            if docs_processed >= target_docs: break

            try:
                with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    for doc in content.split('\n\n'):
                        doc = doc.strip()
                        if len(doc) < 10: continue

                        # Пропускаем уже обработанные документы, чтобы дойти до нужного места
                        if docs_skipped < docs_processed:
                            docs_skipped += 1
                            continue

                        # Токенизация и паддинг
                        tokens = tokenizer.encode(doc, add_bos=True)[:max_length]
                        pad_len = max_length - len(tokens)
                        tokens = np.pad(np.array(tokens, dtype=np.int32), (0, pad_len))

                        all_tokens.append(tokens)
                        docs_processed += 1
                        docs_skipped += 1
                        pbar.update(1)

                        # Сохраняем промежуточный результат каждые 50000 документов
                        if docs_processed % 50 == 0:
                            print(f"\n[Checkpoint] Saving at {docs_processed} docs...")
                            np.save(CACHE_FILE, np.stack(all_tokens))

                        if docs_processed >= target_docs: break
            except Exception:
                continue

    print("\nSaving final numpy array on Google Drive...")
    tokens_array = np.stack(all_tokens)
    np.save(CACHE_FILE, tokens_array)
    print(f"✅ Cache saved! Shape: {tokens_array.shape}, Size: {tokens_array.nbytes / (1024**3):.2f} GB")

# Запускаем создание кэша (например, для 1 миллиона документов)
build_tokenized_cache(target_docs=1000)

Reading and tokenizing texts to save in /content/drive/Shareddrives/shared_veriga/tfds_cache/pretokenized_cache/openwebtext_tokens_2048.npy...


Processing documents:   0%|          | 0/1000 [00:00<?, ?it/s]


Saving final numpy array on Google Drive...
✅ Cache saved! Shape: (162, 2048), Size: 0.00 GB


### Использование кэша для Grain/Kauldron
Загружаем датасет с диска. `mmap_mode='r'` означает, что файл не грузится целиком в RAM, но при этом поддерживает честный индексный доступ, который нужен `grain.MapTransform`.

In [55]:
@dataclasses.dataclass(frozen=True)
class CLMTask(grain.MapTransform):
    """Tokenize full document text → fixed-length tokens array for CLM training.

    Loss is computed on every token position — no src/dst masking.
    Suitable for Wikipedia, PG-19, C4, etc.
    """
    in_text: str
    out_tokens: str
    tokenizer: object
    max_length: int

    def map(self, element):
        text = element[self.in_text]
        if isinstance(text, bytes):
            text = text.decode("utf-8")
        token_ids = self.tokenizer.encode(text, add_bos=True)
        token_ids = token_ids[:self.max_length]
        pad_len = self.max_length - len(token_ids)
        import numpy as np
        token_ids = np.pad(np.array(token_ids, dtype=np.int32), (0, pad_len))
        out = dict(element)
        out[self.out_tokens] = token_ids
        return out

class NumpyMapDataset:
    """Идеальная обертка для Kauldron/Grain. Поддерживает честный случайный доступ."""
    def __init__(self, npy_path):
        # Читаем файл с диска без загрузки в оперативную память
        self.data = np.load(npy_path, mmap_mode='r')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        # Возвращаем словарь в формате, который ожидает модель
        # np.array(...) копирует нужную строку в память
        return {'tokens': np.array(self.data[index], dtype=np.int32)}

def get_cached_clm_dataset(batch_size, cache_file=CACHE_FILE):
    if not os.path.exists(cache_file):
        raise FileNotFoundError(f"Cache not found at {cache_file}. Please build it first.")

    dataset = NumpyMapDataset(cache_file)

    return kd.data.py.DataSource(
        dataset,
        batch_size=batch_size,
        shuffle=True,      # Теперь перемешивание будет работать корректно!
        num_epochs=None,   # Бесконечное обучение
        transforms=[
            CLMTask(
                in_text="text",
                out_tokens="tokens",
                tokenizer=tokenizer,
                max_length=max_length,
            ),
            kd.data.py.Elements(keep=["tokens"]),
        ]
    )

In [56]:
# Clear any existing dataset reference and test the cached dataset
try:
    del ds
except NameError:
    pass

try:
    ds = get_cached_clm_dataset(batch_size)
    print("Starting cached dataset test...")

    it = iter(ds)
    batch = next(it)
    print("✅ Success! Loaded batch with shape:", batch['tokens'].shape)
    print("Sample token IDs:", batch['tokens'][0][:10])
except FileNotFoundError as e:
    print(f"❌ {e}")
    print("Пожалуйста, раскомментируйте 'build_tokenized_cache(...)' в ячейке выше и запустите её для создания кэша.")
except Exception as e:
    print(f"❌ Failed with error: {e}")

❌ Cache not found at /content/drive/Shareddrives/shared_veriga/tfds_cache/pretokenized_cache/openwebtext_tokens_2048.npy. Please build it first.
Пожалуйста, раскомментируйте 'build_tokenized_cache(...)' в ячейке выше и запустите её для создания кэша.


In [51]:
batch.shape

AttributeError: 'dict' object has no attribute 'shape'

In [44]:
# Clear any existing dataset reference and test the new generator
try:
    del ds
except NameError:
    pass

ds = get_clm_dataset(batch_size, tokenizer, max_length)

# Test access: this should now bypass TFDS/Beam and read files directly
print("Starting dataset test...")
try:
    # Using manual iteration to verify the generator works outside of Grain's indexer
    it = iter(ds)
    batch = next(it)
    print("✅ Success! Loaded batch with shape:", batch['tokens'].shape)
    print("Sample token IDs:", batch['tokens'][0][:10])
except Exception as e:
    print(f"❌ Failed with error: {e}")

Starting dataset test...
Disabling pygrain multi-processing (unsupported in colab).
❌ Failed with error: object of type 'generator' has no len()


In [11]:
import apache_beam
import os

# Verification of the manual download path
manual_dir = os.path.join(TFDS_DATA_DIR, 'downloads/manual')
print(f"Checking for manual files in: {manual_dir}")

if os.path.exists(manual_dir):
    print("Manual directory found. Files present:", os.listdir(manual_dir))
else:
    print("WARNING: Manual directory not found. Please ensure the 'OpenWebText.zip' is in the correct path.")

try:
    # We use take(1) to trigger the generation/loading process
    for batch in ds.take(1):
        print("Successfully accessed the first batch.")
except Exception as e:
    print(f"\nERROR: The dataset failed to initialize. \nDetail: {e}")
    print("\nTroubleshooting Tip: If you see 'ManualDownloadError', ensure the ZIP contains the internal folder structure: OpenWebText/Version 1/URLs/*.txt")

Checking for manual files in: /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/manual
Manual directory found. Files present: ['OpenWebText.zip', '.ipynb_checkpoints']
Disabling pygrain multi-processing (unsupported in colab).
**************************** WARNING *********************************
yet no `beam_runner` nor `beam_options` was explicitly provided.

Some Beam datasets take weeks to generate, so are usually not suited
for single machine generation. Please have a look at the instructions
to setup distributed generation:

https://www.tensorflow.org/datasets/beam_datasets#generating_a_beam_dataset
**********************************************************************


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]


ERROR: The dataset failed to initialize. 
Detail: Unable to pickle fn CallableWrapperDoFn(<function Filter.<locals>.<lambda> at 0x7c93f161bce0>): maximum recursion depth exceeded. User code must be serializable (picklable) for distributed execution. This usually happens when lambdas or closures capture non-serializable objects like file handles, database connections, or thread locks. Try: (1) using module-level functions instead of lambdas, (2) initializing resources in setup() methods, (3) checking what your closure captures.

Troubleshooting Tip: If you see 'ManualDownloadError', ensure the ZIP contains the internal folder structure: OpenWebText/Version 1/URLs/*.txt


In [8]:
import shutil
import os

# 1. Define paths
source_dir = "/content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/manual/Version 1"
temp_root = "/content/tmp_repack"
nested_dir = "/content/tmp_repack/OpenWebText"
output_filename = "/content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/manual/OpenWebText"

# 2. Create the required nested structure: OpenWebText/Version 1/...
if os.path.exists(source_dir):
    if os.path.exists(temp_root): shutil.rmtree(temp_root)
    os.makedirs(nested_dir)

    print(f"Copying files to create nested structure...")
    # Copy 'Version 1' into 'OpenWebText'
    shutil.copytree(source_dir, os.path.join(nested_dir, "Version 1"))

    print(f"Archiving into {output_filename}.zip...")
    # Pack the 'OpenWebText' folder from inside temp_root
    shutil.make_archive(output_filename, 'zip', temp_root, 'OpenWebText')

    print(f"\u2705 Success! Please delete the folder '/content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted' and re-run the dataset cell.")
else:
    print(f"Error: Source directory {source_dir} not found. Please check your Drive mount.")

Error: Source directory /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/manual/Version 1 not found. Please check your Drive mount.


In [9]:
import os

# Let's search for the actual location of 'Version 1' or 'URLs' on the Shared Drive
search_root = "/content/drive/Shareddrives/shared_veriga/"
print(f"Searching for dataset files in: {search_root}")

found = False
for root, dirs, files in os.walk(search_root):
    if "Version 1" in dirs or "URLs" in dirs:
        print(f"\n[FOUND] Target files located at: {root}")
        found = True
        # List contents to verify structure
        sub_path = os.path.join(root, "Version 1" if "Version 1" in dirs else "URLs")
        print(f"Contents of {sub_path}: {os.listdir(sub_path)[:5]}")

if not found:
    print("\n[NOT FOUND] Could not find 'Version 1' or 'URLs'. Please ensure the download completed and the Drive is mounted correctly.")

Searching for dataset files in: /content/drive/Shareddrives/shared_veriga/

[FOUND] Target files located at: /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted/ZIP.OpenWebText.zip/OpenWebText
Contents of /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted/ZIP.OpenWebText.zip/OpenWebText/Version 1: ['URLs']

[FOUND] Target files located at: /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted/ZIP.OpenWebText.zip/OpenWebText/Version 1
Contents of /content/drive/Shareddrives/shared_veriga/tfds_cache/downloads/extracted/ZIP.OpenWebText.zip/OpenWebText/Version 1/URLs: ['RS_v2_2007-04.xz.deduped.txt', 'RS_2012-03.bz2.deduped.txt', 'RS_v2_2009-08.xz.deduped.txt', 'RS_2011-11.bz2.deduped.txt', 'RS_v2_2010-03.xz.deduped.txt']


True

## 5. Оптимизатор (сниженный LR для дообучения)

In [ ]:
from adam_atan2 import adam_atan2
from optax.contrib._muon import MuonDimensionNumbers

def muon_only_dims(params):
    MUON_KEYS = {'to_queries', 'to_keys_values', 'combine_heads'}
    def _label(path, v):
        key    = str(path[-1].key) if hasattr(path[-1], 'key') else ''
        parent = str(path[-2].key) if len(path) > 1 and hasattr(path[-2], 'key') else ''
        if key == 'kernel' and parent in MUON_KEYS:
            return MuonDimensionNumbers(reduction_axis=0, output_axis=1)
        return None
    return jax.tree_util.tree_map_with_path(_label, params)

def is_muon_param(path_str):
    parts = path_str.split('/')
    return (len(parts) >= 2 and
            parts[-1] == 'kernel' and
            parts[-2] in {'to_queries', 'to_keys_values', 'combine_heads'})

def muon_mask(params):
    def _m(path, v):
        return is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def is_gate_param(path_str):
    # Проверяем, относится ли параметр к гейту
    return 'memory_gate' in path_str.split('/')

def gate_mask(params):
    def _m(path, v):
        return is_gate_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def adam_base_mask(params):
    def _m(path, v):
        path_str = '/'.join(str(p.key) for p in path)
        return not is_muon_param(path_str) and not is_gate_param(path_str)
    return jax.tree_util.tree_map_with_path(_m, params)



# Сниженные LR: веса уже предобучены в Phase 1
# lr_muon = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_muon = optax.cosine_decay_schedule(
    init_value=1e-5,
    decay_steps=total_steps,
    alpha=0.05,  # конечный lr = 1e-3 * 0.05 = 5e-5
)
# lr_adam = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_adam = optax.cosine_decay_schedule(
    init_value=1e-5,
    decay_steps=total_steps,
    alpha=0.05,
)

lr_gate = optax.cosine_decay_schedule(
    init_value=1e-3,  # В 100 раз больше, чем lr_adam (1e-5)
    decay_steps=total_steps,
    alpha=0.05,
)

## lr for experiments 4e-4, 1.5e-4
inner_chain = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.masked(
        optax.contrib.muon(
            learning_rate=1e-6,
            muon_weight_dimension_numbers=muon_only_dims,
            # beta=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32,
        ),
        mask=muon_mask,
    ),
    # 2. Adam для гейтов (с высоким LR)
    optax.masked(
        adam_atan2(
            learning_rate=1e-3,
            # b1=0.90, b2=0.90,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=gate_mask,
    ),
    # 3. Adam для остальных параметров (с обычным LR)
    optax.masked(
        adam_atan2(
            learning_rate=1e-5,
            # b1=0.90,
            # b2=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=adam_base_mask,
    ),
    )

routing_optimizer = optax.MultiSteps(
    kd.optim.partial_updates(
        inner_chain,
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=16,
)

## 6. Loss и метрики

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss — основной сигнал Phase 2
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# Метрики
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, path_prefix=""):
            if hasattr(tree, 'items'):
                for key, val in tree.items():
                    current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                    if key == "memory_gate":
                        mean_val = jnp.mean(val)
                        openness = jax.nn.sigmoid(mean_val)
                        stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                        stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    else:
                        find_gates(val, current_path)
        find_gates(params)
        return self.State(gate_metrics=flax.core.freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))

train_metrics = {
    "memory_gate": MemoryGateMetric(),
    "huber_delta": HuberDeltaMetric(),
}

## 3. Модель (training_phase=2)

In [ ]:
# !git pull

In [ ]:
# # import gemma_titans
# import gemma_titans
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
# from titans_ckpts import SkipTitans

In [ ]:
# gemma_config = dataclasses.replace(
#     Gemma3_1B_Titans.config,
#     sliding_window_size=128,
#     training_phase=2,
#     titans_layer_indices=active_titans_layers,
#     titans_phase2_first_layer=titans_phase2_first_layer,
# )

# model = Gemma3_1B_Titans(
#     config=gemma_config,
#     dtype=jnp.bfloat16,
#     return_last_only=False,
#     tokens="batch.tokens",
# )

# print(f"Model: titans_layer_indices={gemma_config.titans_layer_indices}, phase2_first={gemma_config.titans_phase2_first_layer}")

## 7. Trainer

In [57]:
workdir_name = f'titans_workdir_phase2_from{titans_phase2_first_layer}'

trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath(f'./{workdir_name}'),
    train_ds=get_cached_clm_dataset(
        batch_size=batch_size
    ),
    model=model,
    # init_transform=FullParamsInit(merged_params),
    num_train_steps=total_steps,
    train_losses=train_losses,
    train_metrics=train_metrics,
    optimizer=routing_optimizer,
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),
)

print(f"Trainer initialized. workdir: {workdir_name}")

FileNotFoundError: Cache not found at /content/drive/Shareddrives/shared_veriga/tfds_cache/pretokenized_cache/openwebtext_tokens_2048.npy. Please build it first.

## 8. TensorBoard

Если у вас уже запущено длительное обучение trainer.train() в ячейке, и нужно запустить заново Tensorboard, вам нужно воспользоваться Терминалом Colab:

terminal command available for runtime restart:
```
fuser -k 6006/tcp && tensorboard --logdir /content/Titans_jax/titans_workdir_phase2_from11/ --port 6006 &
```
после чего обновить Tensorboard


In [ ]:
%reload_ext tensorboard
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()



In [ ]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./{workdir_name}/ --port=6006

## 9. Обучение

In [ ]:
state, aux = trainer.train()


In [ ]:
trainer = trainer.replace(num_train_steps=55000)

state, aux = trainer.train()

## 10. Сохранение весов

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    _, titans_params = titans_tree_utils.split_titans_params(state.params)
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved Titans weights to {save_path}")
new_weights_name = f"saved_titans_phase2_from_{titans_phase2_first_layer}"

save_titans_weights(state, f"./{new_weights_name}")

shutil.make_archive(new_weights_name, 'zip', f"./{new_weights_name}")
print(f"\u2705 {new_weights_name}.zip ready")
destination_path = f"/content/drive/Shareddrives/shared_veriga/saved_titans_delta/{new_weights_name}.zip"
shutil.copy(f"./{new_weights_name}.zip", destination_path)
print(f"Файл скопирован в: {destination_path}")